In [1]:
import pandas as pd
from tc5061_proyecto_equipo17.config import PROCESSED_DATA_DIR, EXTERNAL_DATA_DIR

archivos = sorted(PROCESSED_DATA_DIR.glob("*_procesado.parquet"))
viajes = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
estaciones = pd.read_csv(EXTERNAL_DATA_DIR / "nomenclatura_2026_09.csv")

print(viajes.shape, estaciones.shape)

2026-10-17 18:46:56.306 | INFO     | tc5061_proyecto_equipo17.config:<module>:11 - PROJ_ROOT path is: C:\Users\diego\TC5061-Proyecto-Equipo17


(752314, 10) (484, 5)


In [2]:
estaciones["dpcapacity"].describe()

count    484.000000
mean      16.882231
std        4.126942
min        0.000000
25%       15.000000
50%       15.000000
75%       19.000000
max       39.000000
Name: dpcapacity, dtype: float64

In [3]:
ids_con_viajes = set(viajes["origen_id"]) | set(viajes["destino_id"])
cap_cero = estaciones[estaciones["dpcapacity"] == 0]

print("Estaciones con capacidad 0:", len(cap_cero))
cap_cero[cap_cero["id"].isin(ids_con_viajes)]

Estaciones con capacidad 0: 4


,id,name,latitude,longitude,dpcapacity


In [4]:
salidas = (
    viajes.assign(hora_ts=viajes["inicio_del_viaje"].dt.floor("h"))
    .groupby(["origen_id", "hora_ts"]).size()
    .rename("salidas")
)
salidas.index.names = ["estacion_id", "hora_ts"]

llegadas = (
    viajes.assign(hora_ts=viajes["fin_del_viaje"].dt.floor("h"))
    .groupby(["destino_id", "hora_ts"]).size()
    .rename("llegadas")
)
llegadas.index.names = ["estacion_id", "hora_ts"]

print(salidas.head(), llegadas.head(), sep="\n\n")

estacion_id  hora_ts            
2            2025-01-01 08:00:00    1
             2025-01-01 09:00:00    1
             2025-01-01 10:00:00    1
             2025-01-01 16:00:00    1
             2025-01-01 18:00:00    3
Name: salidas, dtype: int64

estacion_id  hora_ts            
2            2025-01-01 06:00:00    2
             2025-01-01 10:00:00    1
             2025-01-01 12:00:00    1
             2025-01-01 14:00:00    1
             2025-01-01 17:00:00    1
Name: llegadas, dtype: int64


In [5]:
horas = pd.date_range("2025-01-01 00:00", "2025-02-28 23:00", freq="h")
idx = pd.MultiIndex.from_product(
    [sorted(ids_con_viajes), horas], names=["estacion_id", "hora_ts"]
)

flujo = (
    pd.concat([salidas, llegadas], axis=1)
    .fillna(0)
    .reindex(idx, fill_value=0)
    .astype(int)
)
flujo["neto"] = flujo["llegadas"] - flujo["salidas"]

print(flujo.shape)
flujo.loc[2].head(20)

(521088, 3)


,salidas,llegadas,neto
hora_ts,,,
2025-01-01 00:00:00,0,0,0
2025-01-01 01:00:00,0,0,0
2025-01-01 02:00:00,0,0,0
2025-01-01 03:00:00,0,0,0
2025-01-01 04:00:00,0,0,0
2025-01-01 05:00:00,0,0,0
2025-01-01 06:00:00,0,2,2
2025-01-01 07:00:00,0,0,0
2025-01-01 08:00:00,1,0,-1


In [6]:
cap = estaciones.set_index("id")["dpcapacity"]
est = flujo.index.get_level_values("estacion_id")
dia = flujo.index.get_level_values("hora_ts").normalize()

flujo["capacidad"] = est.map(cap)
flujo["neto_acum_dia"] = flujo.groupby([est, dia])["neto"].cumsum()
flujo["ocupacion"] = 0.5 * flujo["capacidad"] + flujo["neto_acum_dia"]
flujo["pct_ocupacion"] = flujo["ocupacion"] / flujo["capacidad"]
flujo["riesgo_ahora"] = (
    (flujo["pct_ocupacion"] < 0.10) | (flujo["pct_ocupacion"] > 0.90)
).astype(int)
flujo["riesgo_siguiente_hora"] = flujo.groupby(level="estacion_id")["riesgo_ahora"].shift(-1)

print(flujo["riesgo_siguiente_hora"].value_counts(normalize=True))
flujo.loc[2].iloc[6:20]

riesgo_siguiente_hora
0.0    0.844377
1.0    0.155623
Name: proportion, dtype: float64


,salidas,llegadas,neto,capacidad,neto_acum_dia,ocupacion,pct_ocupacion,riesgo_ahora,riesgo_siguiente_hora
hora_ts,,,,,,,,,
2025-01-01 06:00:00,0,2,2,15,2,9.5,0.633333,0,0.0
2025-01-01 07:00:00,0,0,0,15,2,9.5,0.633333,0,0.0
2025-01-01 08:00:00,1,0,-1,15,1,8.5,0.566667,0,0.0
2025-01-01 09:00:00,1,0,-1,15,0,7.5,0.500000,0,0.0
2025-01-01 10:00:00,1,1,0,15,0,7.5,0.500000,0,0.0
2025-01-01 11:00:00,0,0,0,15,0,7.5,0.500000,0,0.0
2025-01-01 12:00:00,0,1,1,15,1,8.5,0.566667,0,0.0
2025-01-01 13:00:00,0,0,0,15,1,8.5,0.566667,0,0.0
2025-01-01 14:00:00,0,1,1,15,2,9.5,0.633333,0,0.0


In [7]:
hora = flujo.index.get_level_values("hora_ts")
flujo["hora_dia"] = hora.hour
flujo["dia_semana"] = hora.dayofweek
flujo["fin_de_semana"] = (flujo["dia_semana"] >= 5).astype(int)
flujo["neto_ult_3h"] = (
    flujo.groupby(level="estacion_id")["neto"]
    .transform(lambda s: s.rolling(3, min_periods=1).sum())
)

flujo[["hora_dia", "dia_semana", "neto", "neto_ult_3h"]].loc[2].iloc[6:12]

,hora_dia,dia_semana,neto,neto_ult_3h
hora_ts,,,,
2025-01-01 06:00:00,6,2,2,2.0
2025-01-01 07:00:00,7,2,0,2.0
2025-01-01 08:00:00,8,2,-1,1.0
2025-01-01 09:00:00,9,2,-1,-2.0
2025-01-01 10:00:00,10,2,0,-2.0
2025-01-01 11:00:00,11,2,0,-1.0


In [8]:
from sklearn.metrics import f1_score, precision_score, recall_score

datos = flujo.dropna(subset=["riesgo_siguiente_hora"])
fechas = datos.index.get_level_values("hora_ts")
test = datos[fechas >= "2025-02-01"]

y_test = test["riesgo_siguiente_hora"].astype(int)
pred_baseline = test["riesgo_ahora"]

print("test_precision:", precision_score(y_test, pred_baseline))
print("test_recall:   ", recall_score(y_test, pred_baseline))
print("test_f1:       ", f1_score(y_test, pred_baseline))

test_precision: 0.7943491997721034
test_recall:    0.7921132143687214
test_f1:        0.7932296313536858


In [9]:
estaciones["dpcapacity"].describe()

count    484.000000
mean      16.882231
std        4.126942
min        0.000000
25%       15.000000
50%       15.000000
75%       19.000000
max       39.000000
Name: dpcapacity, dtype: float64

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

FEATURES = [
    "hora_dia", "dia_semana", "fin_de_semana", "capacidad",
    "pct_ocupacion", "riesgo_ahora", "neto", "neto_ult_3h",
]

train = datos[fechas < "2025-02-01"]
X_train = train[FEATURES]
y_train = train["riesgo_siguiente_hora"].astype(int)
X_test = test[FEATURES]

modelo = RandomForestClassifier(
    n_estimators=100, max_depth=12, class_weight="balanced",
    random_state=42, n_jobs=-1,
)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)
proba = modelo.predict_proba(X_test)[:, 1]

print("test_precision:", precision_score(y_test, pred))
print("test_recall:   ", recall_score(y_test, pred))
print("test_f1:       ", f1_score(y_test, pred))
print("test_pr_auc:   ", average_precision_score(y_test, proba))

test_precision: 0.7140719790129529
test_recall:    0.8997495028794257
test_f1:        0.7962292178483689
test_pr_auc:    0.9057205762665158


In [11]:
%load_ext autoreload
%autoreload 2
from tc5061_proyecto_equipo17.features import p2_flujo_horario

flujo_fn = p2_flujo_horario(viajes, "2025-01-01", "2025-02-28")
pd.testing.assert_frame_equal(flujo_fn, flujo[["salidas", "llegadas", "neto"]])
print("✅ Idéntico al notebook", flujo_fn.shape)

✅ Idéntico al notebook (521088, 3)


In [12]:
from tc5061_proyecto_equipo17.features import p2_etiqueta_riesgo, p2_ocupacion_estimada

f = p2_etiqueta_riesgo(p2_ocupacion_estimada(flujo_fn, estaciones))
cols = ["capacidad", "neto_acum_dia", "ocupacion", "pct_ocupacion",
        "riesgo_ahora", "riesgo_siguiente_hora"]
pd.testing.assert_frame_equal(f[cols], flujo[cols])
print("✅ Idéntico al notebook", f.shape)

✅ Idéntico al notebook (521088, 9)


In [13]:
from tc5061_proyecto_equipo17.features import P2_FEATURES, P2_OBJETIVO, p2_construir_dataset

ds = p2_construir_dataset(viajes, estaciones, "2025-01-01", "2025-02-28")
pd.testing.assert_frame_equal(ds[P2_FEATURES], datos[P2_FEATURES])
assert (ds[P2_OBJETIVO] == datos[P2_OBJETIVO].astype(int)).all()
assert P2_OBJETIVO not in P2_FEATURES
print("✅ Pipeline completo idéntico", ds.shape)

✅ Pipeline completo idéntico (520720, 13)
